# Cleaning a PostgreSQL Database
![Clean PostgreSQL Database](Project_Image.jpeg)

In this project, you will work with data from a hypothetical Super Store to challenge and enhance your SQL skills in data cleaning. This project will engage you in identifying top categories based on the highest profit margins and detecting missing values, utilizing your comprehensive knowledge of SQL concepts.

## Data Dictionary:

### `orders`:
| Column | Definition | Data type | Comments |
|--------|------------|-----------|----------|
| `row_id`| Unique Record ID | `INTEGER` |
| `order_id` | Identifier for each order in table | `TEXT` | Connects to `order_id` in `returned_orders` table |
| `order_date` | Date when order was placed | `TEXT` |
| `market` | Market order_id belongs to | `TEXT` |
| `region` | Region Customer belongs to | `TEXT` | Connects to `region` in `people` table |
| `product_id` | Identifier of Product bought | `TEXT` | Connects to `product_id` in `products` table |
| `sales` | Total Sales Amount for the Line Item | `DOUBLE PRECISION` |
| `quantity` | Total Quantity for the Line Item | `DOUBLE PRECISION` |
| `discount` | Discount applied for the Line Item | `DOUBLE PRECISION` |
| `profit` | Total Profit earned on the Line Item | `DOUBLE PRECISION` |

### `returned_orders`:
| Column | Definition | Data type |
|--------|------------|-----------|
| `returned`| Yes values for Order / Line Item Returned | `TEXT` |
| `order_id` | Identifier for each order in table | `TEXT` |
| `market` | Market order_id belongs to | `TEXT` |

### `people`:
| Column | Definition | Data type |
|--------|------------|-----------|
| `person`| Name of Salesperson credited with Order | `TEXT` |
| `region` | Region Salesperson in operating in | `TEXT` |

### `products`:
| Column | Definition | Data type |
|--------|------------|-----------|
| `product_id`| Unique Identifier for the Product | `TEXT` |
| `category` | Category Product belongs to | `TEXT` |
| `sub_category` | Sub Category Product belongs to | `TEXT` |
| `product_name` | Detailed Name of the Product | `TEXT` |

As you can see in the Data Dictionary above, date fields have been written to the `orders` table as `TEXT` and numeric fields like sales, profit, etc. have been written to the `orders` table as `Double Precision`. You will need to take care of these types in some of the queries. This project is an excellent opportunity to apply your SQL skills in a practical setting and gain valuable experience in data cleaning and analysis. Good luck, and happy querying!

In [20]:
-- top_five_products_each_category
WITH top_product_each_category AS (
	SELECT p.category AS category, 
		 p.product_name AS product_name, 
		 SUM(o.sales)::NUMERIC AS product_total_sales, 
		 SUM(o.profit)::NUMERIC AS product_total_profit,
		 RANK() OVER(PARTITION BY p.category ORDER BY SUM(o.sales) DESC) AS product_rank
    FROM products as p
    LEFT JOIN orders as o
    USING(product_id)
    GROUP BY p.category, p.product_name
)

-- top_five_products_each_category
SELECT category,
	product_name,
	ROUND(product_total_sales, 2) AS product_total_sales,
	ROUND(product_total_profit, 2) AS product_total_profit,
	product_rank
FROM top_product_each_category
WHERE product_rank <= 5
ORDER BY category

,category,product_name,product_total_sales,product_total_profit,product_rank
0,Furniture,"Hon Executive Leather Armchair, Adjustable",58193.48,5997.25,1
1,Furniture,"Office Star Executive Leather Armchair, Adjust...",51449.80,4925.80,2
2,Furniture,"Harbour Creations Executive Leather Armchair, ...",50121.52,10427.33,3
3,Furniture,"SAFCO Executive Leather Armchair, Black",41923.53,7154.28,4
4,Furniture,"Novimex Executive Leather Armchair, Adjustable",40585.13,5562.35,5
5,Office Supplies,"Eldon File Cart, Single Width",39873.23,5571.26,1
6,Office Supplies,"Hoover Stove, White",32842.60,-2180.63,2
7,Office Supplies,"Hoover Stove, Red",32644.13,11651.68,3
8,Office Supplies,"Rogers File Cart, Single Width",29558.82,2368.82,4
9,Office Supplies,"Smead Lockers, Industrial",28991.66,3630.44,5


In [21]:
-- impute_missing_values
WITH unit_price_cte AS (
	SELECT product_id,
		   discount,
		   market,
		   region,
		   SUM(sales)/SUM(quantity::NUMERIC) AS unit_price
	FROM orders 
	WHERE quantity IS NOT NULL
	GROUP BY product_id, discount, market, region
),

imputed_missing_values AS (
	SELECT o.product_id,
	o.discount,
	o.market,
	o.region,
	o.sales,
	o.quantity,
    CASE 
		WHEN quantity IS NULL THEN sales/unit_price
	    ELSE quantity END AS calculated_quantity
FROM orders AS o
LEFT JOIN unit_price_cte AS upc
		ON o.product_id = upc.product_id
        AND o.discount = upc.discount
        AND o.market = upc.market
        AND o.region = upc.region
WHERE quantity IS NULL
)

SELECT *
FROM imputed_missing_values
ORDER BY product_id, discount, market, region;

,product_id,discount,market,region,sales,quantity,calculated_quantity
0,FUR-ADV-10000571,0.00,EMEA,EMEA,438.960,NaN,4.0
1,FUR-ADV-10004395,0.00,EMEA,EMEA,84.120,NaN,2.0
2,FUR-BO-10001337,0.15,US,West,308.499,NaN,3.0
3,TEC-STA-10003330,0.00,Africa,Africa,506.640,NaN,2.0
4,TEC-STA-10004542,0.00,Africa,Africa,160.320,NaN,4.0


In [1]:
SELECT * FROM public.orders

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,city,state,country,postal_code,market,region,product_id,sales,quantity,discount,profit,shipping_cost,order_priority
0,957,MX-2014-105921,2014-05-28,2014-06-03,Standard Class,ZC-21910,Zuschuss Carroll,Consumer,San Salvador,San Salvador,El Salvador,NaN,LATAM,Central,TEC-AC-10004626,342.080,2.0,0.00,0.0000,21.713,Medium
1,24359,ID-2013-61442,2013-01-15,2013-01-21,Standard Class,JB-16000,Joy Bell-,Consumer,Manila,National Capital,Philippines,NaN,APAC,Southeast Asia,OFF-BI-10001400,122.400,5.0,0.15,0.0000,21.710,Low
2,32298,CA-2012-124891,2012-07-31,2012-07-31,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,United States,10024.0,US,East,TEC-AC-10003033,2309.650,7.0,0.00,762.1845,933.570,Critical
3,26341,IN-2013-77878,2013-02-05,2013-02-07,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,Australia,NaN,APAC,Oceania,FUR-CH-10003950,3709.395,9.0,0.10,-288.7650,923.630,Critical
4,25330,IN-2013-71249,2013-10-17,2013-10-18,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,Australia,NaN,APAC,Oceania,TEC-PH-10004664,5175.171,9.0,0.10,919.9710,915.490,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51285,29002,IN-2014-62366,2014-06-19,2014-06-19,Same Day,KE-16420,Katrina Edelman,Corporate,Kure,Hiroshima,Japan,NaN,APAC,North Asia,OFF-FA-10000746,65.100,5.0,0.00,4.5000,0.010,Medium
51286,35398,US-2014-102288,2014-06-20,2014-06-24,Standard Class,ZC-21910,Zuschuss Carroll,Consumer,Houston,Texas,United States,77095.0,US,Central,OFF-AP-10002906,0.444,1.0,0.80,-1.1100,0.010,Medium
51287,40470,US-2013-155768,2013-12-02,2013-12-02,Same Day,LB-16795,Laurel Beltran,Home Office,Oxnard,California,United States,93030.0,US,West,OFF-EN-10001219,22.920,3.0,0.00,11.2308,0.010,High
51288,9596,MX-2012-140767,2012-02-18,2012-02-22,Standard Class,RB-19795,Ross Baird,Home Office,Valinhos,São Paulo,Brazil,NaN,LATAM,South,OFF-BI-10000806,13.440,2.0,0.00,2.4000,0.003,Medium


In [6]:
SELECT * FROM public.products

,product_id,category,sub_category,product_name
0,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...
1,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black"
2,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID"
3,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless"
4,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed"
...,...,...,...,...
10287,OFF-FA-10004112,Office Supplies,Fasteners,"Stockwell Staples, 12 Pack"
10288,OFF-BI-10003253,Office Supplies,Binders,"Ibico Index Tab, Economy"
10289,OFF-BI-10002510,Office Supplies,Binders,"Acco Index Tab, Clear"
10290,FUR-ADV-10002329,Furniture,Furnishings,"Advantus Light Bulb, Erganomic"


In [5]:
SELECT * FROM public.people

,region,person
0,Central,Anna Andreadi
1,South,Chuck Magee
2,East,Kelly Williams
3,West,Matt Collister
4,Africa,Deborah Brumfield
5,EMEA,Larry Hughes
6,Canada,Nicole Hansen
7,Caribbean,Giulietta Dortch
8,Central Asia,Nora Preis
9,North,Jack Lebron


In [1]:
SELECT * FROM public.people

,region,person
0,Central,Anna Andreadi
1,South,Chuck Magee
2,East,Kelly Williams
3,West,Matt Collister
4,Africa,Deborah Brumfield
5,EMEA,Larry Hughes
6,Canada,Nicole Hansen
7,Caribbean,Giulietta Dortch
8,Central Asia,Nora Preis
9,North,Jack Lebron


In [4]:
SELECT * FROM public.returned_orders

,returned,order_id,market
0,Yes,MX-2013-168137,LATAM
1,Yes,US-2011-165316,LATAM
2,Yes,ES-2013-1525878,EU
3,Yes,CA-2013-118311,United States
4,Yes,ES-2011-1276768,EU
...,...,...,...
1168,Yes,ES-2013-2639112,EU
1169,Yes,CA-2014-134194,United States
1170,Yes,ES-2012-3246286,EU
1171,Yes,ES-2012-4379168,EU
